# 01.4 Dataset and DataLoader / 数据集与数据加载器

这一节解决一个非常实际的问题：模型训练时，数据到底是怎么一批一批送进来的。  
This notebook answers a very practical question: how data is fed into a model batch by batch during training.

重点概念 / Key concepts:

- 数据集 / dataset
- 样本 / sample
- 批次 / batch
- 打乱 / shuffle
- 数据加载器 / data loader
- 自定义数据集 / custom dataset

如果这一节没弄清楚，后面的训练循环通常会写得很乱。  
If this notebook is not clear, later training loops often become messy.

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 理解 `Dataset` 和 `DataLoader` 的职责分工 / Understand the different roles of `Dataset` and `DataLoader`.
2. 自己实现一个最小 `Dataset` / Implement a minimal custom `Dataset`.
3. 用 `DataLoader` 生成 batch / Use `DataLoader` to create batches.
4. 理解 `batch_size` 和 `shuffle` 的作用 / Understand the roles of `batch_size` and `shuffle`.
5. 看懂一个 batch 的形状 / Read the shapes inside a batch.
6. 为后续训练循环准备好数据输入接口 / Prepare the data input interface for later training loops.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset

## 1. `Dataset` 和 `DataLoader` 的区别
## The Difference Between `Dataset` and `DataLoader`

最简单的理解方式 / The simplest way to think about them:

- `Dataset`：定义“第 i 个样本是什么” / defines what the i-th sample is
- `DataLoader`：定义“样本如何组成 batch 并被迭代出来” / defines how samples are batched and iterated

也可以记成 / You can also remember it like this:

- `Dataset` 负责内容 / `Dataset` is responsible for content
- `DataLoader` 负责组织和运输 / `DataLoader` is responsible for organization and delivery

## 2. 用内置 `TensorDataset` 快速感受
## A Quick Start with Built-in `TensorDataset`

先不自己实现，先用 `TensorDataset` 感受最小流程。  
Before implementing a custom dataset, we start with `TensorDataset` to feel the minimal workflow.

In [ ]:
X = torch.tensor(
    [
        [1.0, 0.5],
        [2.0, 1.0],
        [3.0, 1.5],
        [4.0, 2.0],
        [5.0, 2.5],
        [6.0, 3.0],
    ],
    dtype=torch.float32,
)
y = torch.tensor([0, 0, 0, 1, 1, 1], dtype=torch.long)

dataset = TensorDataset(X, y)

print("样本数 / number of samples:", len(dataset))
print("第 0 个样本 / sample 0:", dataset[0])
print("第 3 个样本 / sample 3:", dataset[3])

In [ ]:
loader = DataLoader(dataset, batch_size=2, shuffle=False)

for batch_idx, (xb, yb) in enumerate(loader):
    print(f"batch {batch_idx}")
    print("xb =\n", xb)
    print("yb =", yb)
    print("xb.shape =", xb.shape)
    print("yb.shape =", yb.shape)
    print()

这里最重要的是看形状 / The most important thing here is the shapes:

- 单个样本特征 / one sample feature shape: `(2,)`
- 一个 batch 的特征 / one batch feature shape: `(batch_size, 2)`
- 一个 batch 的标签 / one batch label shape: `(batch_size,)`

也就是说，`DataLoader` 自动在最前面加上了 batch 维。  
That is, `DataLoader` automatically adds the batch dimension in front.

In [ ]:
# 练习 1 / Exercise 1
# 用上面的 dataset 创建一个 DataLoader。
# Create a DataLoader from the dataset above.
#
# 要求 / Requirements:
# 1. batch_size=3
# 2. shuffle=False
# 3. 打印每个 batch 的 xb.shape 和 yb.shape

# loader_ex =
# for xb, yb in loader_ex:
#     print(xb.shape, yb.shape)

In [ ]:
# 练习 1 参考答案 / Exercise 1 Reference Solution

loader_ex = DataLoader(dataset, batch_size=3, shuffle=False)
for xb, yb in loader_ex:
    print(xb.shape, yb.shape)

## 3. 自定义 `Dataset` / Building a Custom `Dataset`

真正做项目时，你通常不会只用 `TensorDataset`，而是要自己定义数据如何被读取。  
In real projects, you usually need to define how data is read yourself, instead of only using `TensorDataset`.

一个最小自定义 `Dataset` 只需要实现两件事：  
A minimal custom `Dataset` only needs two things:

- `__len__`
- `__getitem__`

In [ ]:
class SimpleTabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

        assert len(self.features) == len(self.labels), (
            "features 和 labels 长度必须一致 / features and labels must have the same length"
        )

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        x = self.features[index]
        y = self.labels[index]
        return x, y


features = [
    [1.0, 0.2],
    [2.0, 0.4],
    [3.0, 0.7],
    [4.0, 0.9],
]
labels = [0, 0, 1, 1]

custom_ds = SimpleTabularDataset(features, labels)
print("len(custom_ds) =", len(custom_ds))
print("custom_ds[2] =", custom_ds[2])

这个类做了几件重要的事 / This class does a few important things:

- 把输入整理成张量 / converts inputs to tensors
- 保证特征和标签长度一致 / ensures features and labels have the same length
- 让每次索引都返回一个样本 / returns one sample per index

In [ ]:
# 练习 2 / Exercise 2
# 实现一个 Dataset，要求返回字典格式：
# Implement a Dataset that returns a dict:
# {"features": x, "label": y}

class DictDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, index):
        # TODO
        pass


# ds = DictDataset(features, labels)
# print(len(ds))
# print(ds[0])

In [ ]:
# 练习 2 参考答案 / Exercise 2 Reference Solution

class DictDatasetSolution(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return {
            "features": self.features[index],
            "label": self.labels[index],
        }


ds = DictDatasetSolution(features, labels)
print(len(ds))
print(ds[0])

## 4. `batch_size` 和 `shuffle`
## `batch_size` and `shuffle`

这两个参数几乎每次训练都会碰到。  
These two arguments appear in almost every training workflow.

- `batch_size`：每次喂给模型多少样本 / how many samples go into the model at once
- `shuffle`：每个 epoch 是否打乱顺序 / whether to shuffle sample order each epoch

训练集通常设 `shuffle=True`，验证集和测试集通常设 `False`。  
Training sets usually use `shuffle=True`, while validation and test sets usually use `False`.

In [ ]:
torch.manual_seed(42)

loader_no_shuffle = DataLoader(custom_ds, batch_size=2, shuffle=False)
loader_shuffle = DataLoader(custom_ds, batch_size=2, shuffle=True)

print("不打乱 / no shuffle")
for xb, yb in loader_no_shuffle:
    print(xb[:, 0], yb)

print()
print("打乱 / shuffle")
for xb, yb in loader_shuffle:
    print(xb[:, 0], yb)

## 5. 一个 batch 到底长什么样 / What a Batch Actually Looks Like

理解 batch 的形状，是后续看懂模型输入输出的基础。  
Understanding batch shapes is foundational for reading model inputs and outputs later.

一般来说 / In general:

- 单样本特征 / single-sample features: `(num_features,)`
- batch 特征 / batch features: `(batch_size, num_features)`
- 单样本标签 / single label: `()` 或 `(1,)`
- batch 标签 / batch labels: `(batch_size,)`

In [ ]:
loader = DataLoader(custom_ds, batch_size=3, shuffle=False)
xb, yb = next(iter(loader))

print("xb =\n", xb)
print("yb =", yb)
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)

In [ ]:
# 练习 3 / Exercise 3
# 用 custom_ds 创建一个 DataLoader，batch_size=4。
# Create a DataLoader from custom_ds with batch_size=4.
#
# 取出第一个 batch，并打印：
# Take the first batch and print:
# 1. xb.shape
# 2. yb.shape
# 3. xb[0]
# 4. yb[0]

# loader_big =
# xb, yb = next(iter(loader_big))
# print(xb.shape)
# print(yb.shape)
# print(xb[0])
# print(yb[0])

In [ ]:
# 练习 3 参考答案 / Exercise 3 Reference Solution

loader_big = DataLoader(custom_ds, batch_size=4, shuffle=False)
xb, yb = next(iter(loader_big))
print(xb.shape)
print(yb.shape)
print(xb[0])
print(yb[0])

## 6. 综合小例子 / Integrated Mini Example

下面把 `Dataset`、`DataLoader` 和遍历 batch 串起来。  
Now we combine `Dataset`, `DataLoader`, and batch iteration into one small example.

In [ ]:
train_loader = DataLoader(custom_ds, batch_size=2, shuffle=True)

for step, (xb, yb) in enumerate(train_loader):
    batch_mean = xb.mean(dim=0)
    print(f"step={step}")
    print("xb.shape =", xb.shape)
    print("yb.shape =", yb.shape)
    print("batch mean / batch 均值 =", batch_mean)
    print()

## 7. 小结 / Summary

本节最关键的区分是：  
The most important distinction in this notebook is:

- `Dataset` 定义样本 / `Dataset` defines samples
- `DataLoader` 组织 batch / `DataLoader` organizes batches

你现在应该能回答 / You should now be able to answer:

1. 为什么 `Dataset` 只需要 `__len__` 和 `__getitem__`？ / Why does a minimal `Dataset` only need `__len__` and `__getitem__`?
2. `batch_size` 会怎样影响 batch 的 shape？ / How does `batch_size` affect batch shapes?
3. 为什么训练集常常要 `shuffle=True`？ / Why do training sets often use `shuffle=True`?
4. 为什么模型通常接收的是 batch，而不是单个样本？ / Why do models usually receive batches rather than single samples?

下一步建议 / Suggested next step:

- 进入 `nn.Module` notebook，开始真正定义网络结构 / Move to the `nn.Module` notebook and start defining actual network structures.